# Pass 1 — Extract Objectives from Every Column

**Goal:** For each fund, send ALL non-empty objective columns in a single API call.  
The LLM extracts objectives **per column independently** — no cross-column reasoning yet.  

**Output:** One row per fund, with per-column extraction results stored as JSON.

In [1]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])

MODEL = "claude-sonnet-4-6"  # UPDATE as needed

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

In [2]:
PASS1_SYSTEM_PROMPT = """You are extracting fund objectives from regulatory disclosure text for European mutual funds.

You will receive MULTIPLE columns of text for a single fund. Extract objectives from EACH column INDEPENDENTLY.
Do NOT cross-reference between columns. Treat each column as a standalone source.

WHAT IS A FUND OBJECTIVE:
The fund objective is the statement of what the fund aims to achieve for its investors — its goal or intended outcome.
Examples: long-term capital growth, regular income, maximizing total returns, beating a benchmark, matching an index.

MULTIPLE OBJECTIVES:
There may be more than one objective per column. Extract ALL and label them separately.
Split objectives when joined by "and", "while", "which also", "that also", or similar.
If one part states a financial goal and another states a sustainability goal, split them.
Examples:
- "provide income and moderate capital growth" → two objectives
- "exceed the performance of the index while maintaining a higher ESG score" → two objectives
- "seek long-term capital growth while reducing the risk of capital loss" → two objectives

Sustainability objectives (extract as separate objectives):
- Reducing greenhouse gas emissions, increasing biodiversity, improving living standards, advancing UN SDGs
- "while maintaining a higher ESG score than the index" or "lower carbon intensity" = separate objective

DO NOT INCLUDE:
- Investment policy/strategy: what the fund invests in, how securities are selected, asset allocation
- Mechanism through which objective is achieved ("by investing in...")
- Types of companies invested in ("invest in companies that...", "companies whose products...")
- Even if the sentence says "sustainable investment objective", extract only the fund's own intended outcome, not company activities
- "while taking into account ESG criteria" or "taking into account the risk level" = not an objective
- SFDR boilerplate: "promotes environmental and/or social characteristics" (Article 8/9 language)
  But DO extract if it promotes specific OUTCOMES (e.g. "reduced carbon emissions")
- Risk information, distribution/dividend policy
- Benchmark references used solely for comparison (not as a target to beat)
- Duplicate objectives within the same column

TIME HORIZON: If stated, include it (e.g. "over a rolling five-year period").

EXTRACTION RULES:
- Extract text VERBATIM from the source — do not paraphrase
- Classify each objective as "financial" or "sustainable"
- If no objective can be identified in a column, return an empty list for that column
- Detect the language of each column and record it
- If the column is non-English, ALSO provide an English translation of each extracted objective

OUTPUT FORMAT:
Return a JSON object where each key is the exact column name, and the value is:
{
  "language": "English" or "French" or "German" etc.,
  "objectives": [
    {
      "objective_text": "exact verbatim text from source",
      "objective_text_english": "English translation (same as objective_text if already English)",
      "objective_type": "financial" or "sustainable"
    }
  ]
}

If a column has no identifiable objective:
{
  "language": "English",
  "objectives": []
}
"""

In [3]:
PASS1_FEW_SHOT = [
    {
        "fund_name": "Example Multi-Column Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities of companies the main business of which is financial services.",
            "PRIIPS KID Objective - French": "Le Fonds vise à maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds et à investir d'une manière conforme aux principes de l'investissement environnemental, social et de gouvernance (ESG). Le Fonds investit à l'échelle mondiale au moins 70 % de son actif total dans les titres de participation de sociétés dont l'activité principale est les services financiers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_text_english": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_type": "financial"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus",
                        "objective_text_english": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainability Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve capital growth and to outperform the benchmark. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions. The fund also aims to have long-term positive impact on environment and social objectives."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "objective_text_english": "achieve capital growth",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "outperform the benchmark",
                        "objective_text_english": "outperform the benchmark",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "contribute to reducing greenhouse gas emissions",
                        "objective_text_english": "contribute to reducing greenhouse gas emissions",
                        "objective_type": "sustainable"
                    },
                    {
                        "objective_text": "have long-term positive impact on environment and social objectives",
                        "objective_text_english": "have long-term positive impact on environment and social objectives",
                        "objective_type": "sustainable"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Norwegian Fund",
        "columns": {
            "PRIIPS KID Objective": "Målsetting\n\nFondets målsetting er å skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK).\n\nFondet skal investere i selskaper globalt som har løsninger på FN's bærekraftsmål og dermed bidrar til omstillingen til et mer bærekraftig samfunn."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "Norwegian",
                "objectives": [
                    {
                        "objective_text": "skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK)",
                        "objective_text_english": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example No-Objective Fund",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index, solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": []
            }
        }
    },
    {
        "fund_name": "Example Company-Activity Exclusion Fund",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "KIID Objective/Investment Policy": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "provide capital growth over the long term (5 years or more)",
                        "objective_text_english": "provide capital growth over the long term (5 years or more)",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    }
]

In [4]:
def get_nonempty_columns(row, objective_columns):
    """Return dict of only columns that have real content (skip empty/NA)."""
    columns = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns[col] = str(value)
    return columns


def pass1_extract(fund_name, fund_id, columns_dict):
    """Send all non-empty columns for one fund; get per-column extractions back."""
    if not columns_dict:
        return {"_error": "No non-empty columns available"}

    columns_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in columns_dict.items()]
    )

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

{columns_text}"""

    messages = []
    for ex in PASS1_FEW_SHOT:
        ex_text = "\n\n".join(
            [f"=== Column: {col} ===\n{val}" for col, val in ex["columns"].items()]
        )
        messages.append({"role": "user", "content": f"Fund Name: {ex['fund_name']}\n\n{ex_text}"})
        messages.append({"role": "assistant", "content": json.dumps(ex["response"], indent=2)})

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=4000,
            temperature=0,
            system=PASS1_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            if "```json" in text:
                return json.loads(text.split("```json")[1].split("```")[0].strip())
            elif "```" in text:
                return json.loads(text.split("```")[1].split("```")[0].strip())
            return {"_error": f"JSON parse error: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [5]:
# === LOAD DATA ===
print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  Total funds: {len(df)}, Columns: {len(df.columns)}")

Loading data...
  Total funds: 5680, Columns: 133


In [6]:
# === RUN PASS 1 ===
# Adjust sample size as needed: df.sample(n=10, random_state=0) for quick test
df_sample = df.sample(n=100, random_state=32)

pass1_results = []

for idx in tqdm(range(len(df_sample)), desc="Pass 1 — Extract"):
    row = df_sample.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Name']

    columns_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)
    result = pass1_extract(fund_name, fund_id, columns_dict)

    pass1_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'columns_sent': list(columns_dict.keys()),
        'num_columns_sent': len(columns_dict),
        'pass1_raw': result  # full per-column JSON
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass1_df = pd.DataFrame(pass1_results)
print(f"\nPass 1 complete: {len(pass1_df)} funds processed")

Pass 1 — Extract:   1%|          | 1/100 [00:21<35:43, 21.65s/it]

   [MS INVF Global Brands Eq Inc Z] tokens — in: 11084, out: 1827


Pass 1 — Extract:   2%|▏         | 2/100 [00:37<29:37, 18.14s/it]

   [DWS Global Value LD] tokens — in: 8521, out: 1434


Pass 1 — Extract:   3%|▎         | 3/100 [00:43<20:44, 12.83s/it]

...

   [WealthInv Qblue Bal GlbAkt AnsTran I] tokens — in: 3825, out: 385


Pass 1 — Extract:  99%|█████████▉| 99/100 [31:17<00:10, 10.51s/it]

   [Ethos Aktiefond A Utdelande (SEK)] tokens — in: 3919, out: 416


Pass 1 — Extract: 100%|██████████| 100/100 [31:31<00:00, 18.91s/it]

   [Quaero Capital Cullen US Value X USD] tokens — in: 8330, out: 1071
   Error for Quaero Capital Cullen US Value X USD: Expecting ',' delimiter: line 46 column 236 (char 1854)

Pass 1 complete: 100 funds processed


In [7]:
# === FLATTEN FOR INSPECTION ===
# Create a human-readable summary alongside the raw JSON

summary_rows = []
for _, row in pass1_df.iterrows():
    raw = row['pass1_raw']
    if '_error' in raw:
        summary_rows.append({
            'FundId': row['FundId'],
            'Fund_Name': row['Fund_Name'],
            'total_objectives_found': 0,
            'columns_with_objectives': 0,
            'error': raw['_error']
        })
        continue

    total_obj = 0
    cols_with_obj = 0
    for col_name, col_data in raw.items():
        if isinstance(col_data, dict) and 'objectives' in col_data:
            n = len(col_data['objectives'])
            total_obj += n
            if n > 0:
                cols_with_obj += 1

    summary_rows.append({
        'FundId': row['FundId'],
        'Fund_Name': row['Fund_Name'],
        'num_columns_sent': row['num_columns_sent'],
        'total_objectives_found': total_obj,
        'columns_with_objectives': cols_with_obj,
        'error': None
    })

summary_df = pd.DataFrame(summary_rows)
print("PASS 1 SUMMARY:")
print(f"  Funds processed: {len(summary_df)}")
print(f"  Funds with errors: {summary_df['error'].notna().sum()}")
print(f"  Funds with ≥1 objective: {(summary_df['total_objectives_found'] > 0).sum()}")
print(f"  Avg objectives per fund: {summary_df['total_objectives_found'].mean():.1f}")
print(f"  Avg columns with objectives: {summary_df['columns_with_objectives'].mean():.1f}")

PASS 1 SUMMARY:
  Funds processed: 100
  Funds with errors: 4
  Funds with ≥1 objective: 94
  Avg objectives per fund: 8.2
  Avg columns with objectives: 6.7


In [8]:
# === SAVE PASS 1 OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Save the raw results (pass1_raw as JSON string for portability)
output_df = pass1_df.copy()
output_df['pass1_raw'] = output_df['pass1_raw'].apply(json.dumps)
output_df['columns_sent'] = output_df['columns_sent'].apply(json.dumps)

p1_filename = f'Pass1_Extract_{len(pass1_df)}_funds_{timestamp}.xlsx'
p1_path = os.path.join(OUTPUT_DIR, p1_filename)
output_df.to_excel(p1_path, index=False, engine='openpyxl')
print(f"Saved: {p1_filename}")
print(f"  → Use this file as input to Pass 2")

Saved: Pass1_Extract_100_funds_20260517_2103.xlsx
  → Use this file as input to Pass 2
